In [1]:
print("hi")

hi


# LangGraph — Introduction Through Graph Theory

> **LangGraph applies classic graph data structures to build controlled, stateful, agent-based AI workflows using nodes (functions) and edges (execution rules).**

This notebook builds intuition for LangGraph from first principles — starting with the **Graph data structure**, then mapping every concept directly to LangGraph.

---

## Table of Contents

| # | Topic |
|---|---|
| 1 | What is a Graph? |
| 2 | Core Graph Components — Nodes & Edges |
| 3 | Directed vs Undirected Graphs |
| 4 | Graph Execution Flow |
| 5 | State in Graphs |
| 6 | Graph Concepts → LangGraph Mapping |
| 7 | Why Graphs for Agentic AI? |
| 8 | Linear vs Graph Thinking |
| 9 | Quick Mental Model |
| 10 | Hands-on: Build Your First LangGraph |

---

## 2. Core Graph Components — Nodes & Edges

### 2.1 Node (Vertex)

A **node** represents a **unit of work or computation**.

| Context | What a Node Is |
|---|---|
| General graph theory | A point that holds data or represents an entity |
| **LangGraph** | A **Python function** that takes state as input and returns updated state |

✅ Think of a node as **one step in your AI workflow**:

```
Examples of LangGraph nodes:
  • load_document_node    → reads a file, stores text in state
  • call_llm_node         → sends prompt to Claude, stores response
  • run_tool_node         → executes a tool call, stores result
  • validate_output_node  → checks response quality, stores pass/fail
  • summarise_node        → condenses text, stores summary
```

Every node has the same signature:
```python
def my_node(state: MyState) -> dict:
    # read from state
    # do some work
    # return ONLY the keys that changed
    return {"key": new_value}
```

---

### 2.2 Edge

An **edge** defines how execution moves from one node to another.

| Context | What an Edge Is |
|---|---|
| General graph theory | A relationship or connection between two nodes |
| **LangGraph** | **Control flow** — determines what runs next and under what condition |

✅ Think of an edge as a **decision or transition**:

```
Types of edges in LangGraph:

  Simple edge   →  always go to the next node
                   add_edge("node_a", "node_b")

  Conditional   →  go to different nodes based on state
  edge             add_conditional_edges("node_a", routing_fn)
```

```
Simple edge:
  [classify] ──────────────────► [answer]

Conditional edge:
  [classify] ──── math? ────────► [math_node]    → END
             └─── other? ────────► [general_node] → END
```

---

## 3. Directed vs Undirected Graphs

### Undirected Graph
- Connections work **both ways**
- Example: mutual friendships — if A knows B, then B knows A

```
[A] ───── [B] ───── [C]
      (bidirectional)
```

### Directed Graph ✅ (Used by LangGraph)
- Connections are **one-way**
- Execution moves in a specific direction — from START → END

```
[A] ──►  [B] ──►  [C]
      (one-way flow)
```

LangGraph is built on a **Directed State Graph**:
- Flow has a clear direction
- State is passed forward through nodes
- Cycles (loops back) are possible and intentional

```
LangGraph directed flow with a cycle:

  START ──► [agent] ──── tool needed? ──Yes──► [tools] ──┐
                    └── No ──► END                        │
                    ▲____________________________________ ─┘
                              (cycle / loop back)
```

> **Key rule:** In LangGraph, you always control the direction — nothing runs unless you declare an edge pointing to it.

---

## 4. Graph Execution Flow

### Traditional (Linear) Programs
```
Step 1 → Step 2 → Step 3 → Done
(top-to-bottom, no branching, no loops)
```

### Graph-Based Programs (LangGraph)
Execution is **dynamic** — it can branch, loop, and re-route based on what happens at runtime.

```
Start
  ↓
LLM Node
  ↓
Decision Node ──Yes──► Tool Node ──► LLM Node (again)
      │
      └──No──► End
```

This flow is **not easy** to write in normal procedural code. Graphs make it natural and readable.

### Why this matters for AI agents

An AI agent needs to:
1. Receive a question
2. Decide if it needs a tool
3. Call the tool (if needed)
4. Look at the tool result
5. Decide if it needs another tool — **or** give a final answer
6. Loop until done

This is inherently a **graph** — not a linear chain.

```
LangGraph agent loop:

  START ──► [agent_node]
                │
                ├── has tool_calls? ──Yes──► [tools_node] ──┐
                │                                           │
                └── No ──► END          (loops back) ◄──────┘
```

---

## 5. State in Graphs — The Most Critical Concept

### In Normal Graphs
Nodes may or may not store data. Each node is independent.

### In LangGraph ✅
There is a **single shared state object** — a `TypedDict` — that travels through the entire graph.

```
                  ┌─────────────────────────────┐
                  │         SHARED STATE         │
                  │  { "article": "...",            │
                  │    "summary": "",            │
                  │    "translation": "" }       │
                  └─────────────────────────────┘
                         ↑ read/write ↓
    [summarise_node]  ←──────────────────  [translate_node]
```

### Key Rule

> **Nodes do NOT talk to each other directly.**  
> They communicate ONLY via the shared state.

```python
# Node reads from state, returns ONLY what it changes
def summarise_node(state: MyState) -> dict:
    summary = call_llm(state["article"])    # reads "article" from state
    return {"summary": summary}             # writes only "summary" back
```

### Benefits of shared state

| Benefit | Why it matters |
|---|---|
| **Predictable** | You always know what data is available at each node |
| **Debuggable** | Print the state at any point to inspect the full context |
| **Loopable** | A node can check state and decide to loop or stop |
| **Memory-safe** | No hidden globals; all data flows through one typed container |
| **Checkpointable** | LangGraph can save and restore state between turns |

### State accumulation with `operator.add`

For lists (like chat message histories), use `Annotated[list, operator.add]` to **append** new values rather than replace them:

```python
from typing import Annotated
import operator

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    #                   ↑ each node appends; never replaces
```

In [5]:
lsta=[1,2,3]
lsta=[4,5,6]
print(lsta)

lsta.append(7)
print(lsta)

[4, 5, 6]
[4, 5, 6, 7]


In [4]:
import operator

result = operator.add(5, 3) # Similar to using the + operator, but in function form
print(result)   # 8

8


In [3]:
import operator

list1 = [1, 2, 3]
list2 = [4, 5, 6]

result = list(map(operator.add, list1, list2))
print(result)   # [5, 7, 9]

[5, 7, 9]


In [2]:
from typing import Annotated
import operator

# Step 1: Define Annotated
numbers: Annotated[list[int], operator.add]

# Step 2: Read it using __annotations__
ann = __annotations__['numbers']   # get Annotated object

# Step 3: Extract metadata
reducer = ann.__metadata__[0]      # operator.add

# Step 4: Use it
result = reducer([1, 2], [3])

print(result)

[1, 2, 3]


---

## 6. Graph Theory → LangGraph Mapping
conceptually it’s the same as graph (DS), but purpose + interpretation is different.
| Graph Theory Concept | LangGraph Meaning | Code |
|---|---|---|
| **Node (vertex)** | Python function / runnable | `def my_node(state) → dict` |
| **Edge** | Execution path between nodes | `add_edge("a", "b")` |
| **Directed edge** | One-way workflow step | `add_edge(START, "first_node")` |
| **Conditional edge** | Route based on state | `add_conditional_edges("a", routing_fn)` |
| **Cycle (back-edge)** | Loop — node re-visited | `add_edge("tools", "agent")` |
| **Graph** | The full agent / workflow | `StateGraph(MyState)` |
| **State** | Shared memory between nodes | `class MyState(TypedDict): ...` |
| **Start node** | Entry point | `START` constant |
| **End node** | Termination point | `END` constant |
| **Checkpointer** | State persistence across calls | `MemorySaver()` / `SqliteSaver()` |

```
Graph Theory          LangGraph
─────────────────     ─────────────────────────────────────
Nodes + Edges    →    StateGraph with nodes and edges
Directed path    →    add_edge(START, "node_a")
Conditional      →    add_conditional_edges("node_a", fn)
Cycle            →    add_edge("node_b", "node_a")
State/data       →    TypedDict passed through all nodes
```

---

## 10. Hands-on: Build Your First LangGraph

Time to apply everything from this notebook in real code!

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# HANDS-ON: Build Your First LangGraph
# Demonstrates: State, Nodes, Edges, Conditional routing, Compile & Invoke
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — Install (run once)
# !pip install langgraph

# ─── Imports ─────────────────────────────────────────────────────────────────
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# ─── Step 2: Define State (shared memory across all nodes) ───────────────────
class AgentState(TypedDict):
    input:    str           # original user question
    response: str           # answer built up by nodes
    steps:    Annotated[list[str], operator.add]   # log of what happened

# ─── Step 3: Define Nodes (each is a Python function) ────────────────────────
def greet_node(state: AgentState) -> dict:
    """Node A — acknowledge the input."""
    msg = f"Received your question: '{state['input']}'"
    print(f"[greet_node] {msg}")
    return {"response": msg, "steps": ["greet_node ran"]}

def check_node(state: AgentState) -> dict:
    """Node B — decide how to answer."""
    if "weather" in state["input"].lower():
        answer = "The weather today is sunny ☀️"
    elif "time" in state["input"].lower():
        import datetime
        answer = f"Current time: {datetime.datetime.now().strftime('%H:%M:%S')}"
    else:
        answer = "Great question! I will need to look that up."
    print(f"[check_node] answer = {answer}")
    return {"response": answer, "steps": ["check_node ran"]}

def respond_node(state: AgentState) -> dict:
    """Node C — format the final response."""
    final = f"✅ FINAL ANSWER: {state['response']}"
    print(f"[respond_node] {final}")
    return {"response": final, "steps": ["respond_node ran"]}

# ─── Step 4: Define conditional routing ─────────────────────────────────────
def route_after_check(state: AgentState) -> str:
    """Edge function — returns the name of the next node to visit."""
    if "look that up" in state["response"]:
        return "respond_node"   # generic response, skip straight to end
    return "respond_node"       # same here — extend this for real branches

# ─── Step 5: Build the Graph ─────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("greet_node",   greet_node)
builder.add_node("check_node",   check_node)
builder.add_node("respond_node", respond_node)

# Add edges
builder.add_edge(START,          "greet_node")    # entry point
builder.add_edge("greet_node",   "check_node")    # always go to check
builder.add_conditional_edges(                    # conditional branch
    "check_node",
    route_after_check,
    {"respond_node": "respond_node"}
)
builder.add_edge("respond_node", END)             # exit point

# Compile
graph = builder.compile()
print("✅ Graph compiled successfully!\n")

# ─── Step 6: Invoke the Graph ─────────────────────────────────────────────────
for question in ["What is the weather today?", "What time is it?", "Tell me about AI"]:
    print(f"\n{'─'*55}")
    print(f"Question: {question}")
    initial_state = {"input": question, "response": "", "steps": []}
    result = graph.invoke(initial_state)
    print(f"Steps taken: {result['steps']}")
    print(f"{result['response']}")

✅ Graph compiled successfully!


───────────────────────────────────────────────────────
Question: What is the weather today?
[greet_node] Received your question: 'What is the weather today?'
[check_node] answer = The weather today is sunny ☀️
[respond_node] ✅ FINAL ANSWER: The weather today is sunny ☀️
Steps taken: ['greet_node ran', 'check_node ran', 'respond_node ran']
✅ FINAL ANSWER: The weather today is sunny ☀️

───────────────────────────────────────────────────────
Question: What time is it?
[greet_node] Received your question: 'What time is it?'
[check_node] answer = Current time: 16:33:11
[respond_node] ✅ FINAL ANSWER: Current time: 16:33:11
Steps taken: ['greet_node ran', 'check_node ran', 'respond_node ran']
✅ FINAL ANSWER: Current time: 16:33:11

───────────────────────────────────────────────────────
Question: Tell me about AI
[greet_node] Received your question: 'Tell me about AI'
[check_node] answer = Great question! I will need to look that up.
[respond_node] ✅ FINAL A